In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/hate-speech-comment-viet-nam/FINAL_TRAINING_DATASET_TEENCODE_20251225_151716.csv


In [4]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer

# Load data
df = pd.read_csv('/kaggle/input/done-final-hate-speech/FINAL_TRAINING_DATASET_TEENCODE_20251225_151716.csv')
df = df.dropna(subset=['training_text', 'label'])
df['training_text'] = df['training_text'].astype(str)

# Encode labels
if df['label'].dtype == 'object':
    le = LabelEncoder()
    df['label'] = le.fit_transform(df['label'])

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

# Split data
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['training_text'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

# Tokenize
def tokenize_function(texts):
    return tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

train_encodings = tokenize_function(train_texts)
val_encodings = tokenize_function(val_texts)

# Dataset class
class ToxicDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = ToxicDataset(train_encodings, train_labels)
val_dataset = ToxicDataset(val_encodings, val_labels)

print(f"✅ Train: {len(train_dataset)}, Val: {len(val_dataset)}")

✅ Train: 4267, Val: 1067


In [5]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Tính trọng số dựa trên tập nhãn huấn luyện
# 'balanced' sẽ tự động gán trọng số cao cho nhãn ít và thấp cho nhãn nhiều
weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
class_weights = torch.tensor(weights, dtype=torch.float).to('cuda')

print(f"Trọng số các lớp [0, 1, 2] là: {weights}")
# Giải thích: Trọng số càng cao, model càng bị "phạt" nặng nếu đoán sai lớp đó.

Trọng số các lớp [0, 1, 2] là: [0.79548844 1.12526371 1.17064472]


In [8]:
# ============================================================================
# 0. THIẾT LẬP HỆ THỐNG
# ============================================================================
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================================
# 1. CHUẨN BỊ DỮ LIỆU & TRỌNG SỐ TÙY CHỈNH (BOOST LỚP 1)
# ============================================================================
df = pd.read_csv('/kaggle/input/done-final-hate-speech/FINAL_TRAINING_DATASET_TEENCODE_20251225_151716.csv').dropna(subset=['training_text', 'label'])
df['label'] = df['label'].astype(int)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['training_text'].values, df['label'].values, 
    test_size=0.2, random_state=42, stratify=df['label']
)

# 💡 CHIẾN THUẬT: Ép trọng số thủ công để cứu lớp Offensive (1)
# Thay vì dùng 'balanced', ta tăng trọng số lớp 1 lên 1.25 và lớp 2 là 1.20
custom_weights = [0.8, 1.25, 1.2] 
class_weights = torch.tensor(custom_weights, dtype=torch.float32).to(device)
print(f"🎯 Trọng số lớp tùy chỉnh [Sạch, Công kích, Thù ghét]: {custom_weights}")

# ============================================================================
# 2. TOKENIZER & MODEL (FREEZE KIẾN THỨC NỀN)
# ============================================================================
model_name = "vinai/phobert-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# Đóng băng 6 lớp đầu để tập trung tinh chỉnh 6 lớp cuối (giảm overfitting)
for name, param in model.roberta.encoder.layer[:6].named_parameters():
    param.requires_grad = False

# ============================================================================
# 3. SUPREME TRAINER V2 (WEIGHTED LOSS + SMOOTHING)
# ============================================================================
class SupremeTrainerV2(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels").long()
        outputs = model(**inputs)
        logits = outputs.logits
        
        m = model.module if hasattr(model, 'module') else model
        num_labels = m.config.num_labels

        # CrossEntropy với Weight tùy chỉnh + Label Smoothing
        loss_fct = nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device),
            label_smoothing=0.1 
        )
        loss = loss_fct(logits.view(-1, num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# ============================================================================
# 4. DATASET & METRICS
# ============================================================================
class ToxicDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=256)
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self): return len(self.labels)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=-1)
    return {
        "acc": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

# ============================================================================
# 5. TRAINING ARGUMENTS (CHỐT HẠ KẾT QUẢ)
# ============================================================================
training_args = TrainingArguments(
    output_dir="./phobert_supreme_v2",
    num_train_epochs=6,               # 💡 Giảm epoch vì model đạt đỉnh ở epoch 4-6
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    
    learning_rate=2e-5,               # 💡 Tăng nhẹ LR để thoát hố tối ưu
    lr_scheduler_type="cosine_with_restarts", # 💎 Giúp nhảy F1 ở các giai đoạn cuối
    weight_decay=0.15,
    warmup_ratio=0.1,
    
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    
    fp16=True,
    logging_steps=10,
    save_total_limit=1,
    report_to="none",
    seed=42
)

# ============================================================================
# 6. THI THỨC
# ============================================================================
trainer = SupremeTrainerV2(
    model=model, args=training_args,
    train_dataset=ToxicDataset(train_texts, train_labels),
    eval_dataset=ToxicDataset(val_texts, val_labels),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] # 💎 Ngắt sớm nếu Val Loss tăng
)

print("\n🚀 BẮT ĐẦU HUẤN LUYỆN SUPREME V2 (GPU T4 ×2)...")
trainer.train()

# Predict & Final Report
predictions = trainer.predict(ToxicDataset(val_texts, val_labels))
y_pred = np.argmax(predictions.predictions, axis=-1)
print("\n" + "="*50)
print("📊 BÁO CÁO SAU TỐI ƯU HÓA")
print("="*50)
print(classification_report(val_labels, y_pred, target_names=["Clean (0)", "Offensive (1)", "Hate (2)"], digits=4))

🎯 Trọng số lớp tùy chỉnh [Sạch, Công kích, Thù ghét]: [0.8, 1.25, 1.2]


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



🚀 BẮT ĐẦU HUẤN LUYỆN SUPREME V2 (GPU T4 ×2)...


Epoch,Training Loss,Validation Loss,Acc,F1 Macro
1,0.945500,0.919681,0.580131,0.572488
2,0.866000,0.812951,0.685098,0.683074
3,0.776800,0.791841,0.713215,0.713907
4,0.694600,0.784282,0.714152,0.714421
5,0.751400,0.802400,0.709466,0.708991
6,0.641500,0.791259,0.722587,0.721359



📊 BÁO CÁO SAU TỐI ƯU HÓA
               precision    recall  f1-score   support

    Clean (0)     0.7904    0.7002    0.7426       447
Offensive (1)     0.6268    0.6804    0.6525       316
     Hate (2)     0.7409    0.7993    0.7690       304

     accuracy                         0.7226      1067
    macro avg     0.7194    0.7266    0.7214      1067
 weighted avg     0.7278    0.7226    0.7234      1067



In [12]:
import pandas as pd

# 1. Tạo DataFrame kết quả dự đoán
results_df = pd.DataFrame({
    'text': val_texts,
    'true_label': val_labels,
    'pred_label': y_pred
})

# 2. Lọc ra những câu dự đoán SAI
error_df = results_df[results_df['true_label'] != results_df['pred_label']].copy()

# Map nhãn số sang chữ cho dễ đọc
label_map = {0: "Sạch (0)", 1: "Offensive (1)", 2: "Hate (2)"}
error_df['true_name'] = error_df['true_label'].map(label_map)
error_df['pred_name'] = error_df['pred_label'].map(label_map)

# 3. HÀM SOI LỖI THEO TỪ KHÓA (Ví dụ: tù, xin lỗi, hãm...)
def inspect_errors(keywords=None):
    display_df = error_df
    if keywords:
        display_df = error_df[error_df['text'].str.contains('|'.join(keywords), case=False)]
    
    print(f"❌ Tìm thấy {len(display_df)} câu dự đoán sai trong tập Validation.")
    print("-" * 100)
    for i, row in display_df.head(50).iterrows(): # Xem 20 câu đầu tiên
        print(f"📝 TEXT: {row['text']}")
        print(f"✅ THỰC TẾ: {row['true_name']}  |  ❌ DỰ ĐOÁN: {row['pred_name']}")
        print("-" * 100)

# ============================================================================
# THỰC THI SOI LỖI
# ============================================================================
# Thiện có thể thêm từ khóa muốn check vào danh sách dưới đây
inspect_errors(keywords=['tù', 'bắt', 'xin lỗi', 'nết', 'mỏ', 'ngu'])

❌ Tìm thấy 75 câu dự đoán sai trong tập Validation.
----------------------------------------------------------------------------------------------------
📝 TEXT: phân biệt vùng miền, miệt thị người dân vùng lũ. tàng keng <person> lên tiếng xin lỗi nhưng . quá muộn! </s> mặt mũi thì khối ngờ mà lời nói không hài lòng toàn dân
✅ THỰC TẾ: Offensive (1)  |  ❌ DỰ ĐOÁN: Sạch (0)
----------------------------------------------------------------------------------------------------
📝 TEXT: mạng xã hội không còn vui nữa </s> vì sao giờ ai cũng livestream kol bán hàng wew <person> con mẹ nó làm cái đấy cần cái mã với mồm dẻo, lâu lâu nói ngu tí cũng được, một job mà không cần học hành cm gì hết trời cho gì xài đấy, hoặc dị dị như pt cũng được thì ez money hơn gấp 3 4 5 lần một thằng software engineer 3 4 năm kinh nghiệm thì chả đâm cả người vào ) việc nhẹ lương cao mặt dày là được
✅ THỰC TẾ: Hate (2)  |  ❌ DỰ ĐOÁN: Offensive (1)
----------------------------------------------------------------------

In [11]:
# Danh sách từ lóng ẩn dụ cần đặc biệt chú ý
hard_slangs = ['thùng xốp', 'đăng xuất', 'bóc lịch', 'chục niên', 'nết', 'mỏ', 'xiên', 'xích']

# Lọc các câu chứa từ lóng nhưng đang bị gán nhãn 0 hoặc sai lệch
re_train_candidates = error_df[error_df['text'].str.contains('|'.join(hard_slangs), case=False)]

print(f"🚀 Thiện ơi, có {len(re_train_candidates)} câu 'vùng xám' cần bạn check lại nhãn trước khi Train mẻ cuối!")
print("-" * 50)
print(re_train_candidates[['text', 'true_name', 'pred_name']].head(10))

# Xuất ra file để Huy và Kiệt cùng sửa
# re_train_candidates.to_csv('need_to_fix.csv', index=False)


🚀 Thiện ơi, có 13 câu 'vùng xám' cần bạn check lại nhãn trước khi Train mẻ cuối!
--------------------------------------------------
                                                  text      true_name  \
35   vụ xe phân khối lớn đâm vào đôi bạn trẻ trên p...  Offensive (1)   
72   cay đắng cảnh <person> bắt quả tang vợ lén lút...       Hate (2)   
98   cay đắng cảnh <person> bắt quả tang vợ lén lút...  Offensive (1)   
135  mới đây, mạng xã hội lan truyền clip 1 nhóm bạ...       Hate (2)   
217  phân biệt vùng miền, miệt thị người dân vùng l...       Hate (2)   
335  phân biệt vùng miền, miệt thị người dân vùng l...       Hate (2)   
530  cay đắng cảnh <person> bắt quả tang vợ lén lút...       Sạch (0)   
572  hai thanh niên đại học là lối chửi bới cựu chi...       Sạch (0)   
792  mới đây, mạng xã hội lan truyền clip 1 nhóm bạ...       Hate (2)   
879  mới đây, mạng xã hội lan truyền clip 1 nhóm bạ...       Hate (2)   

         pred_name  
35        Hate (2)  
72        Sạch (0)  
9